google collab depedencies

In [420]:
# !pip -q install bertopic
# !pip -q install sastrawi
# !pip -q install gensim

In [421]:
# !git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
# %cd topic_modeling_KBMI4

In [422]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [423]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : NVIDIA GeForce GTX 1650 Ti


In [424]:
df = pd.read_csv("data/preprocessed_data.csv")
df = df[df['year'] == 2025]
# df = df[df['bank'] == "LIVIN_MANDIRI_REVIEWS"]
df = df[df['bank'] == "BRIMO_REVIEWS"]
# df = df[df['bank'] == "WONDR_BNI_REVIEWS"]
# df = df[df['bank'] == "BCAMOBILE_REVIEWS"]
df.head()

,reviewId,bank,score,year,text
37800,15144646-fd9c-4843-87b2-8c1d79c44685,BRIMO_REVIEWS,1,2025,sekarang topup ewallet malah ribet lol
37801,03e15614-dbbf-4725-930c-e74f258eceba,BRIMO_REVIEWS,1,2025,layanan enggak bagus
37802,dd382476-aca6-4172-bbe0-5c8c48dfc4ac,BRIMO_REVIEWS,1,2025,kenapa setiap buka aplikasi brimo selalu tampi...
37803,c9d73b55-ac20-4739-8d6b-1dc1fbbb20ed,BRIMO_REVIEWS,1,2025,harus verifikasi lagi di mana kondisi ktp suda...
37804,08fcea89-91b6-42bc-89ce-82cc1017a26d,BRIMO_REVIEWS,1,2025,aplikasi eror mulu


In [425]:
print(f"total dokumen sebelum filter: {len(df):,}")

total dokumen sebelum filter: 21,202


In [426]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df = df[df["word_count"] >= 5].reset_index(drop=True)
print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 18,529


In [427]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 18,529


# SIMCSE IndoBERT

In [428]:
from sentence_transformers import SentenceTransformer

# Gunakan SimCSE untuk menekan anisotropy dan merapatkan klaster
embedding_model = SentenceTransformer("LazarusNLP/simcse-indobert-base", device=device)

embeddings = embedding_model.encode(
    documents,
    batch_size=128,             # GPU T4/V100 Colab sanggup menangani batch 128 untuk 60k data
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # Wajib: Memaksa vektor berukuran L2=1 agar Cosine Distance presisi
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/145 [00:00<?, ?it/s]

# BERTopic

In [429]:
# embeddings = np.load("indobert_embeddings.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (18529, 768)


stop words

In [430]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [431]:
from nltk.corpus import stopwords as nltk_stopwords
from bertopic.vectorizers import ClassTfidfTransformer

sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# Pure stopwords gabungan (NLTK + Sastrawi)
pure_stopwords = list(set(nltk_stopwords.words('indonesian')).union(set(sastrawi_stopwords)))

# topic_stopwords = list(set(
#     pure_stopwords + [
#         "brimo",
#         "livin",
#         "mandiri",
#         "bca",
#         "bni",
#         "wondr"
#     ]
# ))


vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=pure_stopwords,
    token_pattern=r"(?u)\b[^\d\W]+\b",
    min_df=1, # Untuk 60k data, min_df=5 efektif membuang kata typo langka
    # max_df=0.80
)

# Strict c-TF-IDF Transformer untuk memotong frequent words antar-klaster
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True,
)

baseline UMAP for testing purpose

In [432]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [433]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

K-Means

In [434]:
# # searching best K value
# reduced_embeddings = umap_model.fit_transform(embeddings)  # pakai UMAP embedding yang sama

# k_range = range(20, 100, 5)
# inertias = []
# silhouettes = []

# for k in k_range:
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     labels = km.fit_predict(reduced_embeddings)
#     inertias.append(km.inertia_)
#     sil = silhouette_score(reduced_embeddings, labels)
#     silhouettes.append(sil)
#     print(f"k={k} -> inertia={km.inertia_:.1f}, silhouette={sil:.4f}")

# fig, ax1 = plt.subplots(figsize=(10,5))
# ax1.plot(k_range, inertias, 'b-o', label='Inertia (Elbow)')
# ax1.set_xlabel('Jumlah Klaster (K)')
# ax1.set_ylabel('Inertia', color='b')

# ax2 = ax1.twinx()
# ax2.plot(k_range, silhouettes, 'r-s', label='Silhouette')
# ax2.set_ylabel('Silhouette Score', color='r')

# plt.title('Elbow Method & Silhouette Score vs K')
# plt.show()

In [435]:
from sklearn.cluster import KMeans

kmeans_model = KMeans(
    n_clusters=20,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=42,
)

In [436]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=kmeans_model,
    ctfidf_model=ctfidf_model,
    verbose=True
)

In [437]:
print(vectorizer_model.min_df)
print(vectorizer_model.max_df)

1
1.0


In [438]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-14 16:29:54,255 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-14 16:29:55,412 - BERTopic - Dimensionality - Completed ✓
2026-08-14 16:29:55,414 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-14 16:29:55,988 - BERTopic - Cluster - Completed ✓
2026-08-14 16:29:55,995 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-14 16:29:56,755 - BERTopic - Representation - Completed ✓


In [439]:
# print(vectorizer_model.get_params())

outliers removal

In [440]:
# reduced_topics = topic_model.reduce_outliers(
#         documents,
#         topics,
#         strategy="embeddings",
#         threshold=0.70,
#         embeddings=embeddings
#     )

In [441]:
# topic_model.update_topics(
#     documents,
#     topics=reduced_topics
# )

# Evaluation for Topic Quality

Basic Statistics

In [442]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,0,1445,0_aplikasi brimo_buka_brimo_brimo menanggapi,"[aplikasi brimo, buka, brimo, brimo menanggapi...",[upgrade terbaru ini malah mempersulit untuk t...
1,1,1376,1_brimo buka_aplikasi brimo_brimo brimo_brimo,"[brimo buka, aplikasi brimo, brimo brimo, brim...","[kenapa aplikasi brimo enggak bisa di buka, ke..."
2,2,1298,2_potongan_biaya_sms_potong,"[potongan, biaya, sms, potong, ribu, admin, ua...",[kenapa sekarang kok kena biaya sms terus ya d...
3,3,1277,3_saldo_gagal saldo_qris_proses,"[saldo, gagal saldo, qris, proses, berkurang, ...",[nge bug waktu transaksi qris transaksi gagal ...
4,4,1233,4_uang_bri_pinjaman_atm,"[uang, bri, pinjaman, atm, saldo, bank, nasaba...",[selama ini saya nyaman menggunakan brimo piha...
5,5,1149,5_update_update update_update mulu_lemot,"[update, update update, update mulu, lemot, di...",[dikit dikit update dikit dikit update terus s...
6,6,1070,6_dibuka_buka_aplikasinya_update buka,"[dibuka, buka, aplikasinya, update buka, updat...",[habis di update kok malah enggak bisa dibuka ...
7,7,1064,7_username_password_pasword_salah,"[username, password, pasword, salah, sandi, us...",[tiap kali mau login padahal username dan pass...
8,8,981,8_daftar_nomor_otp_kode,"[daftar, nomor, otp, kode, sesuai, no, nomor h...",[ini kenapa pas mau daftar nomor hp tidak sesu...
9,9,957,9_aksesibilitas_nonaktifkan_ulang_hapus,"[aksesibilitas, nonaktifkan, ulang, hapus, per...",[habis update ke versi terbaru setelah login t...


In [443]:
num_topics = len(
    topic_info[topic_info["Topic"] != -1]
)

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 20
Outliers            : 0
Outlier Percentage  : 0.00%


Topic Size

In [444]:
topic_info[["Topic","Count"]]

,Topic,Count
0,0,1445
1,1,1376
2,2,1298
3,3,1277
4,4,1233
5,5,1149
6,6,1070
7,7,1064
8,8,981
9,9,957


Top Words

In [445]:
top_10_topics = topic_model.get_topic_info()
top_10_topics = top_10_topics[top_10_topics.Topic != -1].nlargest(10, "Count")

for _, row in top_10_topics.iterrows():
    topic_id = row['Topic']
    doc_count = row['Count']

    print("=" * 80)
    print(f"TOPIC {topic_id} | JUMLAH DOKUMEN: {doc_count}")
    print("=" * 80)

    # Menampilkan word-score pair bawaan BERTopic (c-TF-IDF scores)
    words_with_scores = topic_model.get_topic(topic_id)
    for word, score in words_with_scores:
        print(f"  - {word:<20} : {score:.4f}")
    print()

TOPIC 0 | JUMLAH DOKUMEN: 1445
  - aplikasi brimo       : 0.2984
  - buka                 : 0.2380
  - brimo                : 0.2366
  - brimo menanggapi     : 0.2348
  - buka brimo           : 0.2340
  - menanggapi           : 0.2267
  - apk brimo            : 0.2216
  - brimo nya            : 0.2187
  - muncul               : 0.2158
  - putih                : 0.2138

TOPIC 1 | JUMLAH DOKUMEN: 1376
  - brimo buka           : 0.4309
  - aplikasi brimo       : 0.3888
  - brimo brimo          : 0.3676
  - brimo                : 0.3487
  - brimo dibuka         : 0.3403
  - buka brimo           : 0.3337
  - buka                 : 0.3209
  - dibuka               : 0.3035
  - akun brimo           : 0.3034
  - brimo update         : 0.2873

TOPIC 2 | JUMLAH DOKUMEN: 1298
  - potongan             : 0.4021
  - biaya                : 0.3976
  - sms                  : 0.3561
  - potong               : 0.3060
  - ribu                 : 0.2983
  - admin                : 0.2959
  - uang             

Representative Reviews

In [446]:
# Ambil info topik dan urutkan berdasarkan jumlah dokumen terbesar (kecuali outlier -1)
topic_info = topic_model.get_topic_info()
top_10_topics = topic_info[topic_info.Topic != -1].nlargest(10, "Count")["Topic"].tolist()

print("=== TOP 10 TOPIK PALING REPRESENTATIF ===")

for topic_id in top_10_topics:
    # Ambil ukuran klaster asli
    cluster_size = topic_info.loc[topic_info.Topic == topic_id, "Count"].values[0]

    # Ambil kata kunci utama topik untuk mempermudah pembacaan aspek
    keywords = ", ".join([w for w, _ in topic_model.get_topic(topic_id)[:10]])

    # Ambil dokumen yang secara matematis paling dekat dengan centroid klaster (Bawaan BERTopic)
    rep_docs = topic_model.get_representative_docs(topic_id)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id} | CLUSTER SIZE: {cluster_size}")
    print(f"KEYWORDS : {keywords}")
    print("=" * 120)

    # BERTopic menyimpan maksimum 3 representative docs per topik secara default
    for i, doc in enumerate(rep_docs, 1):
        print(f"{i}. {doc}")

=== TOP 10 TOPIK PALING REPRESENTATIF ===

TOPIC 0 | CLUSTER SIZE: 1445
KEYWORDS : aplikasi brimo, buka, brimo, brimo menanggapi, buka brimo, menanggapi, apk brimo, brimo nya, muncul, putih
1. upgrade terbaru ini malah mempersulit untuk transfer karena selalu muncul demi keamanan dan kenyamanan bertransaksi pastikan untuk menutup aplikasi lain terlebih dahulu setelah itu silakan buka brimo kembali untuk melanjutkan aktivitas tolong ubah secepatnya pengaturan yang mempersulit penguna aplikasi brimo ini jujur ini upgrade terbaru yang paling bikin saya kecewa sekali kembalikan seperti seperti sebelum di upgrade saja
2. ini kenapa min saat tranfer muncul pop up seperti ini demi keamanan dan kenyamanan bertransaksi pastikan untuk menutup aplikasi lain terlebih dahulu setelah itu silakan buka brimo kembali untuk melanjutkan aktivitasmu saya sudah cek tidak ada aplikasi selain brimo yang berjalan bahkan saya juga sudah cek aplikasi yang mencurigakan yang berjalan di latar belakang kalau sudah

silhoutte score

In [447]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.3328


In [448]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info["Topic"]:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]

    topic_words.append(words)

flat_words = list(chain.from_iterable(topic_words))

unique_words = len(set(flat_words))
total_words = len(flat_words)

topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.8500


NPMI

In [449]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [450]:
doc.split()

['sesudah',
 'update',
 'versi',
 'terbaru',
 'tiap',
 'mau',
 'transaksi',
 'selalu',
 'ada',
 'pesan',
 'pop',
 'up',
 'silakan',
 'tutup',
 'aplikasi',
 'lainnya',
 'agar',
 'bisa',
 'kembali',
 'bertransaksi',
 'sudah',
 'di',
 'tutup',
 'juga',
 'sudah',
 'restart',
 'hp',
 'sudah',
 'coba',
 'uninstall',
 'dan',
 'install',
 'ulang',
 'sudah',
 'ke',
 'kc',
 'bri',
 'dipatiukur',
 'mereka',
 'nyerah',
 'dan',
 'menyarankan',
 'hp',
 'nya',
 'di',
 'bawa',
 'ke',
 'konter',
 'terus',
 'coba',
 'lagi',
 'ke',
 'kc',
 'jl',
 'ir',
 'h',
 'djuanda',
 'yang',
 'mana',
 'kantor',
 'lebih',
 'besar',
 'sampai',
 'itu',
 'nya',
 'turun',
 'tangan',
 'dan',
 'tetap',
 'masalah',
 'enggak',
 'ke',
 'atasi',
 'sampai',
 'detik',
 'ini',
 'belum',
 'pernah',
 'kecewa',
 'ke',
 'brimo',
 'tapi',
 'sekarang',
 'parah',
 'banget',
 'enggak',
 'bisa',
 'top',
 'up',
 'ewallet']

In [451]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [452]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)

top_n = 10
topic_words = []

for topic in topic_info["Topic"]:

    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
        if word in dictionary.token2id
    ]

    if len(words) >= 2:
        topic_words.append(words)

print(f"Valid Topics for NPMI: {len(topic_words)}")

Valid Topics for NPMI: 20


In [453]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : 0.1041


In [454]:
per_topic = np.array(
    coherence_model.get_coherence_per_topic()
)

overall = coherence_model.get_coherence()

print("Gensim overall :", overall)
print("Mean per-topic :", per_topic.mean())
print("Difference     :", overall - per_topic.mean())

Gensim overall : 0.10412150170569108
Mean per-topic : 0.10412150170569108
Difference     : 0.0


# Evaluation for Clustering Quality


DBCV -> Only if using HDBSCAN method

In [455]:
mask = np.array(topics) != -1
X = topic_model.umap_model.embedding_[mask].astype(np.float64)
labels = np.array(topics)[mask]

dbcv_score = validity_index(X, labels)
print(f"DBCV : {dbcv_score:.4f}")

DBCV : -0.7537


In [456]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

Proporsi bank di keseluruhan korpus (baseline):
bank
BRIMO_REVIEWS    100.0
Name: proportion, dtype: float64

bank   BRIMO_REVIEWS  is_imbalanced  topic_size
topic                                          
0                1.0          False        1445
1                1.0          False        1376
18               1.0          False         438
17               1.0          False         580
16               1.0          False         610
15               1.0          False         663
14               1.0          False         690
13               1.0          False         707
12               1.0          False         812
11               1.0          False         859
10               1.0          False         930
9                1.0          False         957
8                1.0          False         981
7                1.0          False        1064
6                1.0          False        1070
5                1.0          False        1149
4                1.0      

checking outliers

In [457]:
# import itertools

# param_grid = {
#     "min_cluster_size": [30, 50, 75],
#     "min_samples": [10, 15, 20],
#     "cluster_selection_method": ["eom", "leaf"],
# }

# results = []
# combos = list(itertools.product(*param_grid.values()))
# print(f"Total kombinasi: {len(combos)}")

# for mcs, ms, method in combos:
#     hdbscan_test = HDBSCAN(
#         min_cluster_size=mcs, min_samples=ms, metric="euclidean",
#         cluster_selection_method=method, prediction_data=True,
#     )
#     tm = BERTopic(
#         embedding_model=None, calculate_probabilities=False,
#         vectorizer_model=vectorizer_model, umap_model=umap_model,
#         hdbscan_model=hdbscan_test, verbose=False,
#     )
#     tpcs, _ = tm.fit_transform(documents, embeddings)

#     ti = tm.get_topic_info()
#     n_topics = len(ti) - 1
#     outlier_pct = (np.array(tpcs) == -1).sum() / len(tpcs) * 100
#     max_share = ti[ti.Topic != -1]["Count"].max() / len(tpcs) * 100 if n_topics > 0 else 0
#     mask = np.array(tpcs) != -1
#     sil = silhouette_score(tm.umap_model.embedding_[mask], np.array(tpcs)[mask]) if len(set(np.array(tpcs)[mask])) > 1 else float("nan")

#     row = {"min_cluster_size": mcs, "min_samples": ms, "method": method,
#            "topics": n_topics, "outlier_%": round(outlier_pct, 2),
#            "max_topic_share_%": round(max_share, 2), "silhouette": round(sil, 4)}
#     results.append(row)
#     print(row)

# results_df = pd.DataFrame(results).sort_values("outlier_%")
# results_df

# eksperimenting to reduce outlier

In [458]:
# # ============================================================
# # TWO-STAGE CLUSTERING ON HDBSCAN OUTLIERS
# # ============================================================

# import numpy as np
# from bertopic import BERTopic
# from hdbscan import HDBSCAN

# # ------------------------------------------------------------
# # 1. Get original outliers
# # ------------------------------------------------------------

# outlier_mask = np.array(topics) == -1

# outlier_documents = [
#     doc for doc, is_outlier in zip(documents, outlier_mask)
#     if is_outlier
# ]

# outlier_embeddings = embeddings[outlier_mask]

# print("Original documents :", len(documents))
# print("Stage 1 outliers   :", len(outlier_documents))


# # ------------------------------------------------------------
# # 2. Create SECOND HDBSCAN
# # ------------------------------------------------------------

# hdbscan_stage2 = HDBSCAN(
#     min_cluster_size=20,
#     min_samples=5,
#     metric="euclidean",
#     cluster_selection_method="eom",
#     prediction_data=True
# )


# # ------------------------------------------------------------
# # 3. Create SECOND BERTopic
# # ------------------------------------------------------------

# topic_model_stage2 = BERTopic(
#     embedding_model=None,
#     calculate_probabilities=False,
#     vectorizer_model=vectorizer_model,
#     umap_model=umap_model,
#     hdbscan_model=hdbscan_stage2,
#     ctfidf_model=ctfidf_model,
#     verbose=True
# )


# # ------------------------------------------------------------
# # 4. Cluster ONLY the original outliers
# # ------------------------------------------------------------

# stage2_topics, _ = topic_model_stage2.fit_transform(
#     outlier_documents,
#     outlier_embeddings
# )


# # ------------------------------------------------------------
# # 5. Stage 2 results
# # ------------------------------------------------------------

# stage2_topics = np.array(stage2_topics)

# stage2_outliers = np.sum(stage2_topics == -1)
# stage2_clustered = len(stage2_topics) - stage2_outliers

# print("\n" + "=" * 60)
# print("STAGE 2 RESULTS")
# print("=" * 60)

# print(f"Input to Stage 2 : {len(outlier_documents):,}")
# print(f"New clusters     : {len(set(stage2_topics)) - (1 if -1 in stage2_topics else 0):,}")
# print(f"Clustered        : {stage2_clustered:,}")
# print(f"Remaining        : {stage2_outliers:,}")
# print(f"Remaining %      : {stage2_outliers / len(stage2_topics) * 100:.2f}%")

In [459]:
# # ============================================================
# # TOP 10 STAGE-2 TOPICS
# # ============================================================

# topic_info_stage2 = topic_model_stage2.get_topic_info()

# display(
#     topic_info_stage2[
#         topic_info_stage2["Topic"] != -1
#     ].head(10)
# )

In [460]:
# # ============================================================
# # REPRESENTATIVE DOCUMENTS - STAGE 2
# # ============================================================

# for topic in topic_info_stage2[
#     topic_info_stage2["Topic"] != -1
# ]["Topic"].head(10):

#     print("=" * 100)
#     print(f"TOPIC {topic}")
#     print("=" * 100)

#     print("KEYWORDS:")
#     print(
#         [
#             word
#             for word, score
#             in topic_model_stage2.get_topic(topic)[:10]
#         ]
#     )

#     docs_topic = topic_model_stage2.get_representative_docs(topic)

#     print("\nREPRESENTATIVE DOCUMENTS:")

#     for i, doc in enumerate(docs_topic[:3], 1):
#         print(f"{i}. {doc}")

#     print()

In [461]:
# # ============================================================
# # COMBINE STAGE 1 + STAGE 2
# # ============================================================

# import numpy as np

# stage1_topics = np.array(topics)
# stage2_topics = np.array(stage2_topics)

# # Start from original Stage 1 labels
# combined_topics = stage1_topics.copy()

# # Get highest topic ID from Stage 1
# valid_stage1_topics = stage1_topics[stage1_topics != -1]

# next_topic_id = (
#     valid_stage1_topics.max() + 1
#     if len(valid_stage1_topics) > 0
#     else 0
# )

# # ------------------------------------------------------------
# # Remap Stage 2 topics so they don't overlap with Stage 1
# # ------------------------------------------------------------

# stage2_valid_topics = sorted(
#     set(stage2_topics) - {-1}
# )

# stage2_mapping = {
#     old_topic: next_topic_id + i
#     for i, old_topic in enumerate(stage2_valid_topics)
# }

# # ------------------------------------------------------------
# # Replace Stage 1 outliers with Stage 2 cluster labels
# # ------------------------------------------------------------

# stage2_positions = np.where(stage1_topics == -1)[0]

# for position, stage2_topic in zip(
#     stage2_positions,
#     stage2_topics
# ):
#     if stage2_topic != -1:
#         combined_topics[position] = stage2_mapping[stage2_topic]

# # ------------------------------------------------------------
# # RESULT
# # ------------------------------------------------------------

# original_outliers = np.sum(stage1_topics == -1)
# remaining_outliers = np.sum(combined_topics == -1)
# final_clustered = len(combined_topics) - remaining_outliers

# print("=" * 60)
# print("COMBINED TWO-STAGE CLUSTERING")
# print("=" * 60)

# print(f"Original documents : {len(combined_topics):,}")
# print(f"Stage 1 outliers   : {original_outliers:,}")
# print(f"Stage 2 recovered  : {original_outliers - remaining_outliers:,}")
# print(f"Final clustered    : {final_clustered:,}")
# print(f"Final outliers     : {remaining_outliers:,}")
# print(
#     f"Final outlier %    : "
#     f"{remaining_outliers / len(combined_topics) * 100:.2f}%"
# )

# print(f"\nStage 1 topics     : {len(set(stage1_topics) - {-1})}")
# print(f"Stage 2 new topics : {len(stage2_valid_topics)}")
# print(
#     f"Combined topics    : "
#     f"{len(set(combined_topics) - {-1})}"
# )

In [462]:
# print("Combined topics :", len(set(combined_topics) - {-1}))
# print("Combined outliers:", sum(t == -1 for t in combined_topics))

In [463]:
# eval_topics = np.array(combined_topics)

# # ============================================================
# # 1. OUTLIER
# # ============================================================

# outlier_count = np.sum(eval_topics == -1)
# outlier_pct = outlier_count / len(eval_topics) * 100

# print("=" * 60)
# print("TWO-STAGE CLUSTERING EVALUATION")
# print("=" * 60)

# print(f"Topics   : {len(set(eval_topics) - {-1})}")
# print(f"Outliers : {outlier_count:,}")
# print(f"Outlier %: {outlier_pct:.2f}%")


# # ============================================================
# # 2. SILHOUETTE
# # ============================================================

# mask = eval_topics != -1

# silhouette = silhouette_score(
#     embeddings[mask],
#     eval_topics[mask]
# )

# print(f"Silhouette : {silhouette:.4f}")


# # ============================================================
# # 3. TOPIC WORDS
# # ============================================================

# topic_words_combined = []

# # Stage 1 topics
# stage1_valid = sorted(set(stage1_topics) - {-1})

# for topic in stage1_valid:
#     words_scores = topic_model.get_topic(topic)

#     if words_scores:
#         topic_words_combined.append([
#             word
#             for word, score in words_scores[:10]
#         ])


# # Stage 2 topics
# for old_topic in stage2_valid_topics:
#     words_scores = topic_model_stage2.get_topic(old_topic)

#     if words_scores:
#         topic_words_combined.append([
#             word
#             for word, score in words_scores[:10]
#         ])


# # ============================================================
# # 4. TOPIC DIVERSITY
# # ============================================================

# unique_words = len(
#     set(
#         word
#         for words in topic_words_combined
#         for word in words
#     )
# )

# total_words = len(topic_words_combined) * 10

# topic_diversity = (
#     unique_words / total_words
# )

# print(f"Topic Diversity : {topic_diversity:.4f}")


# # ============================================================
# # 5. NPMI
# # ============================================================

# coherence_model = CoherenceModel(
#     topics=topic_words_combined,
#     texts=tokenized_docs,
#     dictionary=dictionary,
#     coherence="c_npmi"
# )

# npmi = coherence_model.get_coherence()

# print(f"NPMI : {npmi:.4f}")

In [464]:
# import time 

# min_cluster_sizes = [20, 30, 40, 50, 60, 75, 100]
# min_samples_list = [1, 3, 5, 10]

# results = []


# # ------------------------------------------------------------
# # HELPER: TOPIC DIVERSITY
# # ------------------------------------------------------------

# def calculate_topic_diversity(model, top_n=10):

#     topic_words = []

#     valid_topics = [
#         topic for topic in model.get_topic_info()["Topic"]
#         if topic != -1
#     ]

#     for topic in valid_topics:

#         words_scores = model.get_topic(topic)

#         if words_scores:
#             words = [
#                 word
#                 for word, score in words_scores[:top_n]
#             ]

#             topic_words.append(words)

#     if not topic_words:
#         return np.nan

#     unique_words = len(
#         set(
#             word
#             for words in topic_words
#             for word in words
#         )
#     )

#     total_words = len(topic_words) * top_n

#     return unique_words / total_words


# # ------------------------------------------------------------
# # HELPER: NPMI
# # ------------------------------------------------------------

# def calculate_npmi(model, tokenized_docs, dictionary, top_n=10):

#     topic_words = []

#     valid_topics = [
#         topic for topic in model.get_topic_info()["Topic"]
#         if topic != -1
#     ]

#     for topic in valid_topics:

#         words_scores = model.get_topic(topic)

#         if words_scores:
#             words = [
#                 word
#                 for word, score in words_scores[:top_n]
#             ]

#             topic_words.append(words)

#     if not topic_words:
#         return np.nan

#     coherence_model = CoherenceModel(
#         topics=topic_words,
#         texts=tokenized_docs,
#         dictionary=dictionary,
#         coherence="c_npmi"
#     )

#     return coherence_model.get_coherence()


# # ------------------------------------------------------------
# # EXPERIMENT LOOP
# # ------------------------------------------------------------

# for min_cluster_size in min_cluster_sizes:

#     for min_samples in min_samples_list:

#         print("\n" + "=" * 80)
#         print(
#             f"TESTING: "
#             f"min_cluster_size={min_cluster_size}, "
#             f"min_samples={min_samples}"
#         )
#         print("=" * 80)

#         start_time = time.time()

#         try:

#             # ------------------------------------------------
#             # HDBSCAN
#             # ------------------------------------------------

#             hdbscan_model_test = HDBSCAN(
#                 min_cluster_size=min_cluster_size,
#                 min_samples=min_samples,
#                 metric="euclidean",
#                 cluster_selection_method="eom",
#                 prediction_data=True
#             )

#             # ------------------------------------------------
#             # BERTopic
#             # ------------------------------------------------

#             topic_model_test = BERTopic(
#                 embedding_model=None,
#                 calculate_probabilities=False,
#                 vectorizer_model=vectorizer_model,
#                 umap_model=umap_model,
#                 hdbscan_model=hdbscan_model_test,
#                 ctfidf_model=ctfidf_model,
#                 verbose=False
#             )

#             # ------------------------------------------------
#             # FIT
#             # ------------------------------------------------

#             test_topics, _ = topic_model_test.fit_transform(
#                 documents,
#                 embeddings
#             )

#             test_topics = np.array(test_topics)

#             # ------------------------------------------------
#             # BASIC STATISTICS
#             # ------------------------------------------------

#             total_docs = len(test_topics)

#             outlier_count = np.sum(test_topics == -1)

#             outlier_pct = (
#                 outlier_count / total_docs * 100
#             )

#             num_topics = len(
#                 set(test_topics) - {-1}
#             )

#             # ------------------------------------------------
#             # SILHOUETTE
#             # ------------------------------------------------

#             valid_mask = test_topics != -1

#             unique_valid_topics = len(
#                 set(test_topics[valid_mask])
#             )

#             if (
#                 unique_valid_topics >= 2
#                 and np.sum(valid_mask) > unique_valid_topics
#             ):

#                 silhouette = silhouette_score(
#                     embeddings[valid_mask],
#                     test_topics[valid_mask]
#                 )

#             else:

#                 silhouette = np.nan

#             # ------------------------------------------------
#             # TOPIC DIVERSITY
#             # ------------------------------------------------

#             topic_diversity = calculate_topic_diversity(
#                 topic_model_test
#             )

#             # ------------------------------------------------
#             # NPMI
#             # ------------------------------------------------

#             npmi = calculate_npmi(
#                 topic_model_test,
#                 tokenized_docs,
#                 dictionary
#             )

#             # ------------------------------------------------
#             # TIME
#             # ------------------------------------------------

#             elapsed = time.time() - start_time

#             # ------------------------------------------------
#             # SAVE
#             # ------------------------------------------------

#             results.append({
#                 "min_cluster_size": min_cluster_size,
#                 "min_samples": min_samples,
#                 "num_topics": num_topics,
#                 "outliers": outlier_count,
#                 "outlier_pct": outlier_pct,
#                 "silhouette": silhouette,
#                 "npmi": npmi,
#                 "topic_diversity": topic_diversity,
#                 "runtime_sec": elapsed
#             })

#             print(
#                 f"Topics       : {num_topics}"
#             )

#             print(
#                 f"Outliers     : "
#                 f"{outlier_count:,} "
#                 f"({outlier_pct:.2f}%)"
#             )

#             print(
#                 f"Silhouette   : "
#                 f"{silhouette:.4f}"
#             )

#             print(
#                 f"NPMI         : "
#                 f"{npmi:.4f}"
#             )

#             print(
#                 f"Topic Div.   : "
#                 f"{topic_diversity:.4f}"
#             )

#             print(
#                 f"Runtime      : "
#                 f"{elapsed:.1f}s"
#             )

#         except Exception as e:

#             print(
#                 f"ERROR: {str(e)}"
#             )

#             results.append({
#                 "min_cluster_size": min_cluster_size,
#                 "min_samples": min_samples,
#                 "num_topics": np.nan,
#                 "outliers": np.nan,
#                 "outlier_pct": np.nan,
#                 "silhouette": np.nan,
#                 "npmi": np.nan,
#                 "topic_diversity": np.nan,
#                 "runtime_sec": np.nan
#             })


# # ------------------------------------------------------------
# # FINAL GIANT TABLE
# # ------------------------------------------------------------

# experiment_results = pd.DataFrame(results)

# experiment_results = experiment_results.sort_values(
#     by="silhouette",
#     ascending=False
# ).reset_index(drop=True)


# print("\n" + "=" * 100)
# print("HDBSCAN PARAMETER SWEEP RESULTS")
# print("=" * 100)

# display(experiment_results)

In [465]:
# from bertopic import BERTopic
# from sklearn.feature_extraction.text import CountVectorizer
# import numpy as np

# topics_array = np.array(topics)
# refined_topics = topics_array.copy()

# skipped_topics = []

# for t in set(topics_array):
#     if t == -1:
#         continue
#     idx = np.where(topics_array == t)[0]
#     if len(idx) < 20:
#         continue

#     sub_embeddings = embeddings[idx]
#     sub_documents = [documents[i] for i in idx]

#     sub_vectorizer = CountVectorizer(
#         ngram_range=(1, 2),
#         stop_words=pure_stopwords,   # tetap pakai stopword list yang sama
#         token_pattern=r"(?u)\b[^\d\W]+\b",
#         min_df=1,                     # <- turunkan drastis, sub-cluster jumlahnya sedikit
#     )

#     sub_umap = UMAP(n_neighbors=10, n_components=5, metric="cosine", min_dist=0.0, random_state=42)
#     sub_hdbscan = HDBSCAN(
#         min_cluster_size=max(10, int(len(idx) * 0.3)),
#         min_samples=5, metric="euclidean",
#         cluster_selection_method="eom", prediction_data=True,
#     )

#     try:
#         sub_model = BERTopic(
#             embedding_model=None, umap_model=sub_umap, hdbscan_model=sub_hdbscan,
#             vectorizer_model=sub_vectorizer, calculate_probabilities=False, verbose=False,
#         )
#         sub_topics, _ = sub_model.fit_transform(sub_documents, sub_embeddings)
#         hidden_outlier_mask = np.array(sub_topics) == -1
#         refined_topics[idx[hidden_outlier_mask]] = -1
#     except Exception as e:
#         skipped_topics.append((t, str(e)))
#         continue

# print(f"Outlier awal (tahap 1)              : {(topics_array == -1).sum():,} ({(topics_array==-1).sum()/len(topics_array):.2%})")
# print(f"Outlier setelah refinement (tahap 2): {(refined_topics == -1).sum():,} ({(refined_topics==-1).sum()/len(refined_topics):.2%})")
# print(f"Topik yang di-skip (gagal sub-cluster): {len(skipped_topics)}")
# if skipped_topics:
#     print(skipped_topics[:5])

In [466]:
# for fraction in [0.10, 0.15, 0.20, 0.30]:
#     refined_topics = topics_array.copy()
    
#     for t in set(topics_array):
#         if t == -1:
#             continue
#         idx = np.where(topics_array == t)[0]
#         if len(idx) < 20:
#             continue
        
#         sub_embeddings = embeddings[idx]
#         sub_documents = [documents[i] for i in idx]
        
#         sub_vectorizer = CountVectorizer(
#             ngram_range=(1, 2), stop_words=pure_stopwords,
#             token_pattern=r"(?u)\b[^\d\W]+\b", min_df=1,
#         )
#         sub_umap = UMAP(n_neighbors=10, n_components=5, metric="cosine", min_dist=0.0, random_state=42)
#         sub_hdbscan = HDBSCAN(
#             min_cluster_size=max(10, int(len(idx) * fraction)),
#             min_samples=5, metric="euclidean",
#             cluster_selection_method="eom", prediction_data=True,
#         )
        
#         try:
#             sub_model = BERTopic(
#                 embedding_model=None, umap_model=sub_umap, hdbscan_model=sub_hdbscan,
#                 vectorizer_model=sub_vectorizer, calculate_probabilities=False, verbose=False,
#             )
#             sub_topics, _ = sub_model.fit_transform(sub_documents, sub_embeddings)
#             hidden_outlier_mask = np.array(sub_topics) == -1
#             refined_topics[idx[hidden_outlier_mask]] = -1
#         except Exception:
#             continue
    
#     pct = (refined_topics == -1).sum() / len(refined_topics) * 100
#     print(f"fraction={fraction} -> outlier setelah refinement: {pct:.2f}%")